# Video Dubbing Pipeline: English → Chinese

Dub English TED-style speeches into Chinese using IndexTTS2.

**Features:**
- Voice cloning (speaker identity preserved)
- Emotion preservation (from original audio)
- Speaker diarization (2-4 speakers)
- Background audio preservation (applause, music)
- Duration-aligned output

**Prerequisites:**
1. Set `HF_TOKEN` and `LLM_API_KEY` in [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb)
2. Accept pyannote model terms: https://huggingface.co/pyannote/speaker-diarization-3.1
3. Accept pyannote segmentation terms: https://huggingface.co/pyannote/segmentation-3.0

## 0. Mount Google Drive & Set Model Cache

All large model files (IndexTTS2, Whisper, Demucs, Pyannote) are cached on Google Drive so they persist across sessions.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/models_cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: Whisper, Pyannote, WhisperX alignment, IndexTTS2
# Only set HF_HOME — HF_HUB_CACHE auto-derives as HF_HOME/hub
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
# PyTorch hub models: Demucs
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")

## 1. Install dependencies

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

In [ ]:
# Clone the repo
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts

In [ ]:
%cd /content/index-tts

# Uninstall conflicting Colab packages + install project deps
!pip uninstall -y torch torchvision torchaudio tensorflow keras tensorboard protobuf 2>/dev/null
!pip install ninja
!pip install -e ".[webui]" --extra-index-url https://download.pytorch.org/whl/cu128

# Dubbing pipeline deps
!apt-get update -qq && apt-get install -y -qq rubberband-cli > /dev/null
!pip install -q demucs whisperx pyrubberband soundfile openai

# Verify
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")

## 2. Download IndexTTS2 checkpoints

Checkpoints are downloaded to `checkpoints/` (local). To cache on Drive, we symlink.

In [ ]:
import os

DRIVE_CKPT = f"{DRIVE_CACHE}/indextts2_checkpoints"
LOCAL_CKPT = "/content/index-tts/checkpoints"

# Download to Drive (persists across sessions)
if not os.path.exists(f"{DRIVE_CKPT}/config.yaml"):
    print("Downloading IndexTTS2 checkpoints to Google Drive (first time only)...")
    !huggingface-cli download IndexTeam/IndexTTS-2 --local-dir "{DRIVE_CKPT}"
else:
    print(f"Checkpoints already cached at {DRIVE_CKPT}")

# Symlink so the pipeline finds them at the expected local path
if os.path.islink(LOCAL_CKPT):
    os.unlink(LOCAL_CKPT)
elif os.path.exists(LOCAL_CKPT):
    import shutil
    shutil.rmtree(LOCAL_CKPT) if os.path.isdir(LOCAL_CKPT) else os.remove(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)
print(f"Symlinked: {LOCAL_CKPT} -> {DRIVE_CKPT}")

# Verify
!ls -la checkpoints/config.yaml

## 3. Configure API keys & settings

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')        # HuggingFace (free) — for speaker diarization
LLM_API_KEY = userdata.get('LLM_API_KEY')  # LLM API key — for translation

# --- LLM provider (uncomment one) ---
LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-chat"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- Dubbing settings ---
NUM_SPEAKERS = 2    # Expected number of speakers (2-4)
USE_FP16 = True     # FP16 inference (faster, less VRAM)

print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"Speakers: {NUM_SPEAKERS}, FP16: {USE_FP16}")

## 4. Pre-download heavy models (optional)

Run this cell once to download Whisper, Demucs, and Pyannote models to Google Drive. Subsequent sessions will reuse cached models and skip downloads.

In [ ]:
# Pre-download Whisper model (~3GB)
import whisperx
print("Loading Whisper model (cached on Drive)...")
_model = whisperx.load_model("large-v2", "cuda", compute_type="float16")
del _model
print("Whisper model cached.")

# Pre-download Demucs model (~80MB)
import demucs.pretrained
print("Loading Demucs model (cached on Drive)...")
_model = demucs.pretrained.get_model("htdemucs")
del _model
print("Demucs model cached.")

# Pre-download Pyannote diarization model
from whisperx.diarize import DiarizationPipeline
print("Loading Pyannote model (cached on Drive)...")
_diarize = DiarizationPipeline(token=HF_TOKEN, device="cuda")
del _diarize
print("Pyannote model cached.")

import torch; torch.cuda.empty_cache()
print("\nAll models cached on Google Drive. Future sessions will start faster.")

---
## 5. Run Dubbing — Mode A: Single Video

Upload a video file or specify a Google Drive path.

In [ ]:
from dub_pipeline import dub_video
from google.colab import files
from pathlib import Path

# === Option 1: Upload a video ===
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Option 2: Google Drive path (uncomment below) ===
# video_path = "/content/drive/MyDrive/videos/ted_talk.mp4"

# === Run ===
stem = Path(video_path).stem
output_path = f"/content/{stem}_cn.mp4"

dub_video(
    video_path=video_path,
    output_path=output_path,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_dir="checkpoints",
    num_speakers=NUM_SPEAKERS,
    use_fp16=USE_FP16,
)

# Play result inline
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
# Download the dubbed video
files.download(output_path)

---
## 6. Run Dubbing — Mode B: Batch (Google Drive folder)

Process all videos in an input folder. Output goes to a separate folder.

In [ ]:
from dub_pipeline import dub_batch

INPUT_DIR  = "/content/drive/MyDrive/videos/input"   # <- your input folder
OUTPUT_DIR = "/content/drive/MyDrive/videos/output"  # <- dubbed output folder

dub_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    hf_token=HF_TOKEN,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    model_dir="checkpoints",
    num_speakers=NUM_SPEAKERS,
    use_fp16=USE_FP16,
)
print(f"\nAll done! Check: {OUTPUT_DIR}")

---
## 7. Inspect intermediate results

The pipeline saves intermediate files for debugging. Check the `dub_workspace/` directory.

In [ ]:
import json, os
from IPython.display import Audio, display

# Find the most recent work directory
work_dirs = sorted(
    [d for d in os.listdir("dub_workspace") if os.path.isdir(f"dub_workspace/{d}")]
) if os.path.isdir("dub_workspace") else []

if work_dirs:
    work_dir = f"dub_workspace/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    # Show transcript
    transcript_path = f"{work_dir}/transcript.json"
    if os.path.exists(transcript_path):
        with open(transcript_path) as f:
            segments = json.load(f)
        print(f"\n--- Transcript ({len(segments)} segments) ---")
        for s in segments[:10]:
            print(f"  [{s.get('speaker','?')}] {s['start']:.1f}-{s['end']:.1f}s: {s['text']}")
        if len(segments) > 10:
            print(f"  ... and {len(segments)-10} more")

    # Show translations
    trans_path = f"{work_dir}/translations.json"
    if os.path.exists(trans_path):
        with open(trans_path) as f:
            trans = json.load(f)
        print(f"\n--- Translations ---")
        for t in trans[:10]:
            print(f"  #{t['id']} [{t.get('speaker','?')}] EN: {t['en'][:50]}")
            print(f"       ZH: {t['zh'][:50]}")

    # Play separated vocals
    vocals_dir = f"{work_dir}/htdemucs"
    if os.path.isdir(vocals_dir):
        for subdir in os.listdir(vocals_dir):
            voc = f"{vocals_dir}/{subdir}/vocals.wav"
            if os.path.exists(voc):
                print(f"\n--- Separated vocals ---")
                display(Audio(voc))
                break
else:
    print("No work directories found. Run the pipeline first.")